# 02 — Fixed-point policy experiments

This notebook studies the core constrained policy update:
\
\[
\pi = \Gamma_{\Pi}\left(\pi + lpha \, \mathrm{diag}(c'(\pi))\, H_	heta(P)^	op wight),
\]
where the projection is performed on a masked simplex.

In [ ]:
import pathlib
import sys

ROOT = pathlib.Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

import torch
import matplotlib.pyplot as plt

from l2o_cyber_defense import (
    sample_dag_single_sink,
    control_mask_from_dag,
    dag_target_w,
    fixed_point_pi,
    proxy_optimal_policy,
)
from l2o_cyber_defense.utils import theta_pos
from l2o_cyber_defense.visualization import draw_graph_policy, bar_compare_policies

## Build one instance

In [ ]:
torch.manual_seed(1)

P = sample_dag_single_sink(n=20, edge_prob=0.35)
control_mask = control_mask_from_dag(P)
w = dag_target_w(P)

raw_theta = torch.tensor([-1.0, 0.2, 0.8, 1.1, 0.1])
theta = theta_pos(raw_theta)

print("theta =", theta)
print("controllable nodes =", int(control_mask.sum()))
print("target mass =", float(w.sum()))

## Solve the fixed point

In [ ]:
pi_fp = fixed_point_pi(
    P=P,
    theta=theta,
    w=w,
    alpha=0.5,
    K=theta.numel(),
    control_mask=control_mask,
    max_iter=300,
    tol=1e-8,
)

print("Policy mass on control set:", float(pi_fp[control_mask].sum()))
print("Policy mass outside control set:", float(pi_fp[~control_mask].sum()))
print("Policy min / max:", float(pi_fp.min()), float(pi_fp.max()))

In [ ]:
draw_graph_policy(
    P,
    pi_fp,
    title="Fixed-point policy on a single-sink DAG",
    control_mask=control_mask,
)
plt.show()

## Compare with a simple proxy policy

The proxy policy is not the same object as the fixed-point solution, but it gives a useful baseline.

In [ ]:
pi_proxy_hard = proxy_optimal_policy(
    P=P,
    w=w,
    Ksum=20,
    softmax_temp=None,
    control_mask=control_mask,
)

pi_proxy_soft = proxy_optimal_policy(
    P=P,
    w=w,
    Ksum=20,
    softmax_temp=0.5,
    control_mask=control_mask,
)

print("Hard proxy support:", torch.nonzero(pi_proxy_hard > 0, as_tuple=False).flatten().tolist())
print("Soft proxy mass:", float(pi_proxy_soft.sum()))

In [ ]:
bar_compare_policies(pi_fp, pi_proxy_soft, labels=("fixed-point", "proxy-soft"))
plt.show()

In [ ]:
draw_graph_policy(
    P,
    pi_proxy_soft,
    title="Soft proxy policy",
    control_mask=control_mask,
)
plt.show()

## Suggested next step

Move to `03_training_theta_on_dag_dataset.ipynb` to learn the coefficients `theta` over a family of graphs.